# AWS MCP over IAM Roles Anywhere — INSTALL

End-to-end install of a Sympozium **MCPServer** that gives an agent access to the
managed **AWS MCP Server** (`aws___run_script` + docs/discovery tools), authenticated
with **temporary credentials from IAM Roles Anywhere** — no static AWS keys.

This notebook is self-contained. It walks every step from an empty AWS account to a
working, agent-attached MCP server:

1. Parameters & prerequisite checks.
2. Provision AWS resources (CA + client cert, trust anchor, profile, role) — via the
   sibling `provisioning/iam-roles-anywhere-lab.ipynb`.
3. Build the container image **for your cluster node architecture**.
4. Push the image to your registry.
5. Create the certificate Secret in the cluster.
6. Render & apply the MCPServer manifest.
7. Attach the MCP server to your agent.
8. Verify end-to-end (a read-only AWS call through the agent).

> **Read the sibling `README.md` first** for how the pieces fit together and *why*.
> Steps that create AWS resources, push images, or touch the cluster use **your**
> credentials/kubeconfig — run them yourself and review output before continuing.

## 1. Parameters

Adjust these to your environment. Everything downstream derives from them.

**Region**: the AWS MCP Server is only offered in `us-east-1` and `eu-central-1`
(see https://docs.aws.amazon.com/general/latest/gr/aws-mcp.html). Pick one of those
for `MCP_REGION`. Your Roles Anywhere resources should live in the same region.

In [ ]:
import json, subprocess, platform, hashlib
from pathlib import Path

# --- Region / endpoint (AWS MCP: only us-east-1 or eu-central-1) ---
MCP_REGION   = 'eu-central-1'
MCP_ENDPOINT = f'https://aws-mcp.{MCP_REGION}.api.aws/mcp'

# --- Kubernetes ---
K8S_NAMESPACE = 'REPLACE_NAMESPACE'      # namespace where the agent + MCP server live
AGENT_NAME    = 'REPLACE_AGENT'           # agent to attach the MCP server to
PULL_SECRET   = 'REPLACE_PULL_SECRET'     # existing docker-registry Secret in the namespace (private registry)

# --- Container image ---
REGISTRY  = 'REPLACE_REGISTRY'            # e.g. registry.example.com/yournamespace
IMAGE_NAME= 'sympozium-mcp-roles-anywhere'
IMAGE_TAG = '0.1.1'                          # bump on every rebuild
IMAGE     = f'{REGISTRY}/{IMAGE_NAME}:{IMAGE_TAG}'

# --- Pinned build inputs (verified) ---
MCP_PROXY_VERSION = '1.7.0'   # https://pypi.org/project/mcp-proxy-for-aws-cli/
# aws_signing_helper 1.8.5, per AWS docs. The Dockerfile needs the LINUX build
# matching your cluster node arch (filled automatically in the build step).
HELPER = {
  'amd64': ('https://rolesanywhere.amazonaws.com/releases/1.8.5/X86_64/Linux/Amzn2023/aws_signing_helper',
            'beec9ed1c492d93db809890f16713e3556353294b823c2184ad4e891f1b2b54d'),
  'arm64': ('https://rolesanywhere.amazonaws.com/releases/1.8.5/Aarch64/Linux/Amzn2023/aws_signing_helper',
            '3d131aa888cd56da446f9c6bb460b1f0569f6c7edc74eae6193a2fe3928883ba'),
}

HERE = Path.cwd()
print('endpoint :', MCP_ENDPOINT)
print('image    :', IMAGE)
print('namespace:', K8S_NAMESPACE, '  agent:', AGENT_NAME)

## 2. Prerequisites check

Confirms the CLIs are present and that AWS + the cluster are reachable. Nothing is
created here.

In [ ]:
def sh(cmd, check=True):
    print('$', ' '.join(cmd))
    r = subprocess.run(cmd, text=True, capture_output=True)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f'failed: {" ".join(cmd)}')
    return r

for tool in (['podman','--version'], ['kubectl','version','--client','--output=yaml']):
    sh(tool, check=False)

# AWS identity (must be able to create IAM + Roles Anywhere resources)
sh(['aws','sts','get-caller-identity','--output','json'], check=False)
# Cluster reachable
sh(['kubectl','get','ns', K8S_NAMESPACE], check=False)
# Node architecture — the image MUST be built for this
r = sh(['kubectl','get','nodes','-o',
        'jsonpath={range .items[*]}{.metadata.name}{"="}{.status.nodeInfo.architecture}{"\\n"}{end}'], check=False)
print('\n>>> Build the image for the architecture shown above (amd64 or arm64).')

## 3. Provision AWS resources (Roles Anywhere)

The CA + client certificate, trust anchor, profile and IAM role are created by the
sibling notebook **`provisioning/iam-roles-anywhere-lab.ipynb`**. Open and run it now
(its kernel needs `boto3`; `provisioning/pyproject.toml` pins deps). It writes
`provisioning/work/lab-outputs.json` with the ARNs, and the cert/key under
`provisioning/work/easy-rsa/easyrsa3/pki/`.

Set its `REGION` to the same `MCP_REGION` as here, and its `PERMISSION_POLICY_ARN`
to the access scope you want the agent to have (e.g. `job-function/ViewOnlyAccess`
for read-only inventory, or `ReadOnlyAccess` for broader reads).

Once it has run, the cell below loads its outputs.

In [ ]:
LAB = HERE / 'provisioning' / 'work'
outputs = json.loads((LAB / 'lab-outputs.json').read_text())
TRUST_ANCHOR_ARN = outputs['TrustAnchorArn']
PROFILE_ARN      = outputs['ProfileArn']
ROLE_ARN         = list(outputs['Roles'].values())[0]
CERT_PATH        = Path(list(outputs['Certs'].values())[0]['cert'])
KEY_PATH         = Path(list(outputs['Certs'].values())[0]['key'])
assert outputs['Region'] == MCP_REGION, f"lab region {outputs['Region']} != MCP_REGION {MCP_REGION}"
assert CERT_PATH.exists() and KEY_PATH.exists(), 'cert/key missing — run the provisioning notebook'
print('trust anchor:', TRUST_ANCHOR_ARN)
print('profile     :', PROFILE_ARN)
print('role        :', ROLE_ARN)
print('cert        :', CERT_PATH)
print('key         :', KEY_PATH)

## 4. Build the container image (for the cluster node arch)

The image bundles `mcp-proxy-for-aws-cli` (SigV4 proxy) and `aws_signing_helper`
(Roles Anywhere credential helper). It MUST be built for the node architecture
detected in step 2 — a mismatch causes `exec format error` / CrashLoopBackOff.

Set `NODE_ARCH` to `amd64` or `arm64` accordingly.

In [ ]:
NODE_ARCH = 'amd64'   # <-- set to what step 2 reported
helper_url, helper_sha = HELPER[NODE_ARCH]

sh(['podman','build','--no-cache',
    '--platform', f'linux/{NODE_ARCH}',
    '--build-arg', f'MCP_PROXY_VERSION={MCP_PROXY_VERSION}',
    '--build-arg', f'SIGNING_HELPER_URL={helper_url}',
    '--build-arg', f'SIGNING_HELPER_SHA256={helper_sha}',
    '-t', IMAGE,
    str(HERE / 'container')])

# Verify the built architecture BEFORE pushing.
r = sh(['podman','image','inspect', IMAGE, '--format', '{{.Architecture}} {{.Os}}'])
assert r.stdout.split()[0] == NODE_ARCH, f'built arch != {NODE_ARCH}; check cache/platform'
print('image arch OK:', r.stdout.strip())

## 5. Push the image

Uses your existing `podman login` for the registry. Use a versioned **tag** (not a
digest) — some registries do not authorize pull-by-digest with a deploy token.

In [ ]:
sh(['podman','push', IMAGE])

## 6. Create the certificate Secret

Mounts the workload cert/key into the MCP pod at `/certs`. The private key never
leaves your machine except into this Secret.

In [ ]:
sh(['kubectl','create','secret','generic','aws-mcp-certificate',
    '-n', K8S_NAMESPACE,
    f'--from-file=tls.crt={CERT_PATH}',
    f'--from-file=tls.key={KEY_PATH}',
    '--dry-run=client','-o','yaml'])
print('\nReview above, then run the real create:')
# Uncomment to apply:
# sh(['kubectl','create','secret','generic','aws-mcp-certificate','-n',K8S_NAMESPACE,
#     f'--from-file=tls.crt={CERT_PATH}', f'--from-file=tls.key={KEY_PATH}'])

## 7. Render & apply the MCPServer manifest

Fills the `REPLACE_*` placeholders in `k8s/mcpserver.yaml` with the values above and
applies. Then attaches the pull Secret to the ServiceAccount and recreates the pod
so it inherits the pull Secret.

In [ ]:
tpl = (HERE / 'k8s' / 'mcpserver.yaml').read_text()
rendered = (tpl
  .replace('REPLACE_NAMESPACE', K8S_NAMESPACE)
  .replace('REPLACE_REGION', MCP_REGION)
  .replace('REPLACE_TRUST_ANCHOR_ARN', TRUST_ANCHOR_ARN)
  .replace('REPLACE_PROFILE_ARN', PROFILE_ARN)
  .replace('REPLACE_ROLE_ARN', ROLE_ARN)
  .replace('REPLACE_REGISTRY', REGISTRY)
  .replace('REPLACE_TAG', IMAGE_TAG)
  .replace('REPLACE_PULL_SECRET', PULL_SECRET))
out = HERE / 'k8s' / 'mcpserver.rendered.yaml'
out.write_text(rendered)
print('wrote', out)
print(rendered)

In [ ]:
sh(['kubectl','apply','-f', str(HERE / 'k8s' / 'mcpserver.rendered.yaml')])
sh(['kubectl','patch','serviceaccount','aws-mcp','-n',K8S_NAMESPACE,
    '-p', json.dumps({'imagePullSecrets':[{'name':PULL_SECRET}]})])
# recreate the pod so it inherits the pull secret from the SA
sh(['kubectl','delete','pod','-n',K8S_NAMESPACE,'-l','app=aws-mcp'], check=False)

In [ ]:
import time; time.sleep(20)
sh(['kubectl','get','pods','-n',K8S_NAMESPACE,'-l','app=aws-mcp','-o','wide'], check=False)
sh(['kubectl','get','mcpserver','aws-mcp','-n',K8S_NAMESPACE,'-o','wide'], check=False)
print('\nExpect the pod Running (1/1) and MCPServer READY=true.')
print('If ImagePullBackOff: pull secret / tag issue. If CrashLoop exec-format: wrong arch.')

## 8. Attach to the agent and verify end-to-end

Appends the MCP server to the agent's `spec.mcpServers` (preserving existing entries),
then you run a read-only AWS request through the agent.

In [ ]:
# Show current list first (so you can confirm nothing is lost)
sh(['kubectl','get','agent',AGENT_NAME,'-n',K8S_NAMESPACE,
    '-o','jsonpath={.spec.mcpServers}'], check=False)
print()
patch = [{'op':'add','path':'/spec/mcpServers/-',
          'value':{'name':'aws-mcp','toolsPrefix':'aws','timeout':180}}]
sh(['kubectl','patch','agent',AGENT_NAME,'-n',K8S_NAMESPACE,
    '--type=json','-p', json.dumps(patch)])
sh(['kubectl','get','agent',AGENT_NAME,'-n',K8S_NAMESPACE,
    '-o','jsonpath={.spec.mcpServers}'], check=False)

### Final test (run through the agent)

Trigger an AgentRun for the agent with a read-only prompt, e.g.:

> *Use the aws tool (run_script) to list the S3 buckets in the account and show their names.*

**Success** = real bucket names in the answer (chain works, role allows reads).
A write attempt (e.g. *create a bucket*) should return **AccessDenied**, proving
read-only is enforced by the IAM role, not just the proxy flag.

If the agent does not see `aws___*` tools on the first run, wait and retry — the
mcp-bridge discovers the new server when the AgentRun starts.

## Teardown (optional)

- Detach from the agent: remove the `aws-mcp` entry from `spec.mcpServers`.
- `kubectl delete mcpserver aws-mcp -n <ns>` and `kubectl delete secret aws-mcp-certificate configmap aws-mcp-config -n <ns>`.
- AWS resources: run the **Cleanup** cell in `provisioning/iam-roles-anywhere-lab.ipynb`
  (`DO_CLEANUP = True`) to remove the profile, trust anchor and role.
- Local PKI: `rm -rf provisioning/work/` when you no longer need the cert/key.